In [1]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, random_split, Subset

In [2]:
TRAIN_CSV = "../data/kaggle/fashion-mnist_train.csv"
TEST_CSV = "../data/kaggle/fashion-mnist_test.csv"

MODEL_NAME = "transformer_v2"
BATCH_SIZE = 64
LEARNING_RATE = 0.001
NUM_EPOCHS = 100
VALID_RATIO = 0.2
K_FOLDS = 5
RANDOM_SEED = 42



IMAGE_SIZE = 28
PATCH_SIZE = 7
NUM_CLASSES = 10

EMBED_DIM = 32      # down from 64
NUM_HEADS = 2       # down from 4
NUM_LAYERS = 2      # down from 4
MLP_DIM = 64        # down from 128

DROPOUT = 0.1

CLASS_NAMES = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot"
]

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [3]:
class FashionMNISTCSVDataset(Dataset):
    def __init__(self, csv_file):
        df = pd.read_csv(csv_file)

        self.labels = df.iloc[:, 0].to_numpy(dtype=np.int64)
        self.images = df.iloc[:, 1:].to_numpy(dtype=np.float32) / 255.0
        self.images = self.images.reshape(-1, 1, 28, 28)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        image = torch.from_numpy(self.images[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, label

In [4]:
full_train_dataset = FashionMNISTCSVDataset(TRAIN_CSV)
test_dataset = FashionMNISTCSVDataset(TEST_CSV)

print("Full train dataset size:", len(full_train_dataset))
print("Test dataset size:", len(test_dataset))

Full train dataset size: 60000
Test dataset size: 10000


In [5]:
indices = np.arange(len(full_train_dataset))
np.random.seed(RANDOM_SEED)
np.random.shuffle(indices)

fold_indices = np.array_split(indices, K_FOLDS) # split the indices into K_FOLDS equal parts 

for i, fold_idx in enumerate(fold_indices, start=1):
    print(f"Fold {i} size: {len(fold_idx)}")


Fold 1 size: 12000
Fold 2 size: 12000
Fold 3 size: 12000
Fold 4 size: 12000
Fold 5 size: 12000


In [6]:
pin_memory = torch.cuda.is_available()
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=pin_memory)

In [7]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=28, patch_size=7, in_channels=1, embed_dim=64):
        super().__init__()

        assert img_size % patch_size == 0, "Image size must be divisible by patch size."

        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2 # sequence length
        self.patch_dim = in_channels * patch_size * patch_size # 1*7*7

        self.proj = nn.Linear(self.patch_dim, embed_dim)

    def forward(self, x):
        # x: [B, C, H, W]
        B, C, H, W = x.shape

        """
        The unfold (The Slicing)
        The two unfold commands act like a pizza cutter.
         - The first cuts the image into 4 horizontal strips (each 7 pixels high).
        New Shape: [Batch, Channels, 4, 28, 7] 
         - The second cuts those strips into 4 vertical blocks.
        New Shape: [Batch, Channels, 4, 4, 7, 7]
        """
        patches = x.unfold(2, self.patch_size, self.patch_size).unfold(3, self.patch_size, self.patch_size)
        # shape: [B, C, H/patch, W/patch, patch, patch]

        """
        This takes that 4 x 4 grid and lays the squares out in a single straight line.
        """
        patches = patches.contiguous().view(B, C, -1, self.patch_size, self.patch_size)
        # shape: [B, C, num_patches, patch, patch]
        # [B, C, 16, 7, 7]

        patches = patches.permute(0, 2, 1, 3, 4).contiguous()
        # shape: [B, num_patches, C, patch, patch]

        patches = patches.view(B, self.num_patches, -1)
        # shape: [B, num_patches, patch_dim]
        # [B, 16, C*7*7]

        embeddings = self.proj(patches)
        # shape: [B, num_patches, embed_dim]

        return embeddings

In [8]:
class FashionViT(nn.Module):
    def __init__(
        self,
        img_size=28,
        patch_size=7,
        in_channels=1,
        num_classes=10,
        embed_dim=64,
        num_heads=4,
        num_layers=4,
        mlp_dim=128,
        dropout=0.1
    ):
        super().__init__()

        self.patch_embed = PatchEmbedding(
            img_size=img_size,
            patch_size=patch_size,
            in_channels=in_channels,
            embed_dim=embed_dim
        )

        num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embedding = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads, # multi-head attention 
            dim_feedforward=mlp_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers) # stack multiple layers of the encoder

        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        B = x.size(0)

        x = self.patch_embed(x)                         # [B, num_patches, embed_dim]

        cls_tokens = self.cls_token.expand(B, -1, -1)   # [B, 1, embed_dim]
        x = torch.cat((cls_tokens, x), dim=1)           # [B, num_patches+1, embed_dim]

        x = x + self.pos_embedding
        x = self.dropout(x)

        x = self.transformer(x)                         # [B, num_patches+1, embed_dim]
        x = self.norm(x)

        cls_output = x[:, 0]                            # take CLS token
        logits = self.head(cls_output)                  # [B, num_classes]

        return logits

In [9]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total

In [10]:
def evaluate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return running_loss / total, correct / total, np.array(all_preds), np.array(all_labels)

In [11]:
os.makedirs("transformers/saved_models_kfold", exist_ok=True)
os.makedirs("transformers/results_kfold", exist_ok=True)

fold_results = []

for fold in range(K_FOLDS):
    print(f"\n--- Fold {fold + 1}/{K_FOLDS} for {MODEL_NAME} ---")

    val_idx = fold_indices[fold]
    train_idx = np.concatenate([fold_indices[i] for i in range(K_FOLDS) if i != fold])

    train_subset = Subset(full_train_dataset, train_idx)
    val_subset = Subset(full_train_dataset, val_idx)

    train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=pin_memory)
    val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=pin_memory)

    model = FashionViT(
        img_size=IMAGE_SIZE,
        patch_size=PATCH_SIZE,
        in_channels=1,
        num_classes=NUM_CLASSES,
        embed_dim=EMBED_DIM,
        num_heads=NUM_HEADS,
        num_layers=NUM_LAYERS,
        mlp_dim=MLP_DIM,
        dropout=DROPOUT
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    train_losses = []
    train_accuracies = []
    val_losses = []
    val_accuracies = []

    best_val_acc = 0.0
    best_val_acc_epoch = 0

    best_val_loss = float("inf")
    best_val_loss_epoch = 0

    for epoch in range(NUM_EPOCHS):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)

        train_losses.append(train_loss)
        train_accuracies.append(train_acc)
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_val_acc_epoch = epoch + 1
            torch.save(
                model.state_dict(),
                f"transformers/saved_models_kfold/{MODEL_NAME}_fold{fold + 1}_best_acc.pth"
            )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_loss_epoch = epoch + 1
            torch.save(
                model.state_dict(),
                f"transformers/saved_models_kfold/{MODEL_NAME}_fold{fold + 1}_best_loss.pth"
            )

        print(
            f"{MODEL_NAME} | Fold [{fold + 1}/{K_FOLDS}] | Epoch [{epoch + 1}/{NUM_EPOCHS}]"
        )
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")
        print("-" * 50)

    fold_log_df = pd.DataFrame({
        "model": MODEL_NAME,
        "fold": fold + 1,
        "epoch": range(1, NUM_EPOCHS + 1),
        "train_loss": train_losses,
        "train_accuracy": train_accuracies,
        "val_loss": val_losses,
        "val_accuracy": val_accuracies
    })
    fold_log_df.to_csv(f"transformers/results_kfold/{MODEL_NAME}_fold{fold + 1}_training_log.csv", index=False)

    fold_summary = {
        "model": MODEL_NAME,
        "fold": fold + 1,
        "best_val_accuracy": best_val_acc,
        "best_val_accuracy_epoch": best_val_acc_epoch,
        "best_val_loss": best_val_loss,
        "best_val_loss_epoch": best_val_loss_epoch,
        "final_train_loss": train_losses[-1],
        "final_train_accuracy": train_accuracies[-1],
        "final_val_loss": val_losses[-1],
        "final_val_accuracy": val_accuracies[-1]
    }

    fold_results.append(fold_summary)

model_fold_df = pd.DataFrame(fold_results)

model_cv_summary = {
    "model": MODEL_NAME,
    "mean_best_val_accuracy": model_fold_df["best_val_accuracy"].mean(),
    "std_best_val_accuracy": model_fold_df["best_val_accuracy"].std(),
    "mean_best_val_loss": model_fold_df["best_val_loss"].mean(),
    "std_best_val_loss": model_fold_df["best_val_loss"].std(),
    "mean_final_val_accuracy": model_fold_df["final_val_accuracy"].mean(),
    "std_final_val_accuracy": model_fold_df["final_val_accuracy"].std(),
    "mean_final_val_loss": model_fold_df["final_val_loss"].mean(),
    "std_final_val_loss": model_fold_df["final_val_loss"].std()
}

pd.DataFrame([model_cv_summary]).to_csv(
    f"transformers/results_kfold/{MODEL_NAME}_cv_summary.csv", index=False
)


--- Fold 1/5 for transformer_v2 ---
transformer_v2 | Fold [1/5] | Epoch [1/100]
Train Loss: 0.9885 | Train Acc: 0.6443
Val   Loss: 0.6804 | Val   Acc: 0.7377
--------------------------------------------------
transformer_v2 | Fold [1/5] | Epoch [2/100]
Train Loss: 0.6891 | Train Acc: 0.7474
Val   Loss: 0.5617 | Val   Acc: 0.7929
--------------------------------------------------
transformer_v2 | Fold [1/5] | Epoch [3/100]
Train Loss: 0.6060 | Train Acc: 0.7780
Val   Loss: 0.4992 | Val   Acc: 0.8175
--------------------------------------------------
transformer_v2 | Fold [1/5] | Epoch [4/100]
Train Loss: 0.5614 | Train Acc: 0.7969
Val   Loss: 0.4918 | Val   Acc: 0.8132
--------------------------------------------------
transformer_v2 | Fold [1/5] | Epoch [5/100]
Train Loss: 0.5293 | Train Acc: 0.8069
Val   Loss: 0.4528 | Val   Acc: 0.8289
--------------------------------------------------
transformer_v2 | Fold [1/5] | Epoch [6/100]
Train Loss: 0.5134 | Train Acc: 0.8117
Val   Loss: 0.4

In [12]:
EMBED_DIM = 32      # down from 64
NUM_HEADS = 2       # down from 4
NUM_LAYERS = 2      # down from 4
MLP_DIM = 64        # down from 128

In [13]:
EMBED_DIM = 128     # up from 64
NUM_HEADS = 8       # up from 4
NUM_LAYERS = 6      # up from 4
MLP_DIM = 256       # up from 128